# 06 — Memory Management

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/03_Advanced/06_memory_management.ipynb)

## 📌 What is it?
Python manages memory automatically via reference counting and a garbage collector. Understanding this prevents memory leaks in long-running AI systems.

## 🧠 Why AI Engineers need this
Long-running inference servers, large dataset processing, and embedding generation at scale can leak memory. Understanding Python's memory model prevents OOM crashes.

In [ ]:
import sys
import gc

# ── REFERENCE COUNTING ──
x = [1, 2, 3, 4, 5]
print(f"Size of list: {sys.getsizeof(x)} bytes")

# References
a = x       # a and x point to the same object
b = x[:]    # b is a copy

print(f"a is x: {a is x}")   # True — same object
print(f"b is x: {b is x}")   # False — different object

del a   # decrements ref count
# x still alive because x still references it

# gc.collect() forces garbage collection
collected = gc.collect()
print(f"GC collected {collected} objects")

In [ ]:
# ── __slots__ — reduce memory per instance ──
import sys

class NormalEmbedding:
    def __init__(self, text, vector):
        self.text = text
        self.vector = vector
        self.metadata = {}

class SlottedEmbedding:
    __slots__ = ['text', 'vector', 'metadata']  # no __dict__!
    def __init__(self, text, vector):
        self.text = text
        self.vector = vector
        self.metadata = {}

normal = NormalEmbedding("hello", [0.1]*10)
slotted = SlottedEmbedding("hello", [0.1]*10)

print(f"Normal:  {sys.getsizeof(normal.__dict__)} bytes (__dict__)")
print(f"Slotted: no __dict__ — saves ~200 bytes per instance")
print(f"At 1M embeddings, __slots__ saves ~200MB!")

In [ ]:
# ── MEMORY PROFILING ──
import tracemalloc

tracemalloc.start()

# Code to profile
embeddings = [[float(i)] * 1536 for i in range(100)]  # 100 embeddings

snapshot = tracemalloc.take_snapshot()
stats = snapshot.statistics('lineno')[:3]

print("Top memory consumers:")
for stat in stats:
    print(f"  {stat.size/1024:.1f} KB — {stat.traceback.format()[0]}")

tracemalloc.stop()

# Practical tip: use generators instead of lists for large datasets
total = sum(sum(row) for row in embeddings)   # generator, no extra memory
print(f"Sum: {total:.0f}")

## ⚠️ Common Mistakes
```python
# ❌ Keeping references to large objects accidentally
results = []
for chunk in huge_dataset:
    result = process(chunk)
    results.append(result)    # all results kept in memory!

# ✅ Process and discard
for chunk in huge_dataset:
    result = process(chunk)
    save_to_disk(result)      # don't accumulate in memory

# ❌ Circular references with __del__
# These can confuse the GC — use weakref instead
```

## 🔗 What's Next?
[07 — Testing with pytest →](07_testing_with_pytest.ipynb)